<a href="https://colab.research.google.com/github/Khushisoni1224/Sentiment-Analysis-Dashboard/blob/main/Major_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LOADING DATASET
---



In [ ]:
import pandas as pd

# Define the file path
file_path = '/kaggle/twitter_training.csv'

# If the first row looks like data, we add generic names.
column_names = ['ID', 'Entity', 'Sentiment', 'Tweet_Content']
df = pd.read_csv(file_path, names=column_names)

# Display the first few rows to make sure it loaded correctly
df.head()

,ID,Entity,Sentiment,Tweet_Content
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


# DATA CLEANING

In [ ]:
# Remove any rows where the tweet content is empty
df.dropna(subset=['Tweet_Content'], inplace=True)

# Quick look at the 'Entity' column to see what brands/topics are in your data
print(df['Entity'].unique())

['Borderlands' 'CallOfDutyBlackopsColdWar' 'Amazon' 'Overwatch'
 'Xbox(Xseries)' 'NBA2K' 'Dota2' 'PlayStation5(PS5)' 'WorldOfCraft'
 'CS-GO' 'Google' 'AssassinsCreed' 'ApexLegends' 'LeagueOfLegends'
 'Fortnite' 'Microsoft' 'Hearthstone' 'Battlefield'
 'PlayerUnknownsBattlegrounds(PUBG)' 'Verizon' 'HomeDepot' 'FIFA'
 'RedDeadRedemption(RDR)' 'CallOfDuty' 'TomClancysRainbowSix' 'Facebook'
 'GrandTheftAuto(GTA)' 'MaddenNFL' 'johnson&johnson' 'Cyberpunk2077'
 'TomClancysGhostRecon' 'Nvidia']


# Run Sentiment Analysis

In [ ]:
# Install and import VADER
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Function to get the compound score
def calculate_sentiment(text):
    return analyzer.polarity_scores(str(text))['compound']

# Apply the function to your dataset
df['Sentiment_Score'] = df['Tweet_Content'].apply(calculate_sentiment)

# Categorize the scores for easier Power BI charts
def categorize_score(score):
    if score >= 0.05: return 'Positive'
    elif score <= -0.05: return 'Negative'
    else: return 'Neutral'

df['Analysis_Label'] = df['Sentiment_Score'].apply(categorize_score)

df.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.8 MB/s eta 0:00:00


,ID,Entity,Sentiment,Tweet_Content,Sentiment_Score,Analysis_Label
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...,-0.6908,Negative
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,-0.6908,Negative
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,-0.6908,Negative
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,-0.6908,Negative
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,-0.6908,Negative


In [ ]:
def clean_gaming_context(text):
    text = str(text).lower()
    # Replace common gaming 'negative' words with neutral alternatives
    replacements = {
        'murder': 'defeat',
        'kill': 'hit',
        'die': 'lose',
        'dead': 'downed'
    }
    for word, replacement in replacements.items():
        text = text.replace(word, replacement)
    return text

In [ ]:
def categorize_score_refined(score):
    if score >= 0.1:
        return 'Positive'
    elif score <= -0.1:
        return 'Negative'
    else:
        return 'Neutral'

In [ ]:
!pip install vaderSentiment textblob emoji
import emoji
from textblob import TextBlob

# This handles the sentiment, the 'Fact vs Opinion' check, and grabs emojis.
def analyze_social_vibe(text):
    text = str(text)

    # Emotional Tone (VADER)
    score = analyzer.polarity_scores(text)['compound']

    # Fact vs Opinion (TextBlob)
    # Closer to 1.0 means it's a very subjective personal opinion.
    bias = TextBlob(text).sentiment.subjectivity

    # Grab emojis—they add a lot of "standout" context to the dashboard.
    found_emojis = "".join(c for c in text if c in emoji.EMOJI_DATA)

    return score, bias, found_emojis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 10.5 MB/s eta 0:00:00


In [ ]:
df['Cleaned_Content'] = df['Tweet_Content'].apply(clean_gaming_context)

# Applying the advanced metrics to new columns
df[['Score', 'Subjectivity', 'Emojis']] = df['Cleaned_Content'].apply(
    lambda x: pd.Series(analyze_social_vibe(x))
)

In [ ]:
# Using your refined threshold for sentiment and a 0.5 threshold for Post Type.
def final_labels(row):
    # Sentiment Labeling
    if row['Score'] >= 0.1: row['Final_Sentiment'] = 'Positive'
    elif row['Score'] <= -0.1: row['Final_Sentiment'] = 'Negative'
    else: row['Final_Sentiment'] = 'Neutral'

    # Categorizing based on the subjectivity (Opinion vs Fact)
    row['Post_Type'] = 'Opinion' if row['Subjectivity'] > 0.5 else 'Fact/Info'

    return row

df = df.apply(final_labels, axis=1)

In [ ]:
final_columns = [
    'ID', 'Entity', 'Sentiment', 'Tweet_Content',
    'Score', 'Final_Sentiment', 'Subjectivity', 'Post_Type', 'Emojis'
]

df[final_columns].to_csv('social_media_intelligence.csv', index=False)

print("Project prepped! Download 'social_media_intelligence.csv' for your dashboard.")
df[final_columns].head()

Project prepped! Download 'social_media_intelligence.csv' for your dashboard.


,ID,Entity,Sentiment,Tweet_Content,Score,Final_Sentiment,Subjectivity,Post_Type,Emojis
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...,-0.4588,Negative,0.0,Fact/Info,
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,0.0000,Neutral,0.0,Fact/Info,
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,0.0000,Neutral,0.0,Fact/Info,
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,-0.4588,Negative,0.0,Fact/Info,
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,-0.4588,Negative,0.0,Fact/Info,
